In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [2]:
# loading the data
train = pd.read_csv("/home/hossamhamdy/AI_Portfolio/Project_01_House_Price_Prediction/data/train.csv")
test = pd.read_csv("/home/hossamhamdy/AI_Portfolio/Project_01_House_Price_Prediction/data/test.csv")

In [3]:
train.shape,test.shape

((1460, 81), (1459, 80))

In [4]:
y= np.log1p(train['SalePrice'])
train_ID = train['Id']
test_ID = test['Id']
train.drop(['SalePrice', 'Id'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

In [5]:
ntrain = train.shape[0]
ntest = test.shape[0]
ntrain,ntest

(1460, 1459)

In [6]:
#Remove Outliers
#Huge houses with perfect quality but very low price
outlier_idx = train[(train['GrLivArea'] > 4000)].index
print(f"Removing {len(outlier_idx)} outliers")
# Remove from train only
train= train.drop(outlier_idx)
y = y.drop(outlier_idx)
print(f"Train shape after outlier removal: {train.shape}")

Removing 4 outliers
Train shape after outlier removal: (1456, 79)


In [7]:
# apply transformation to all the data
all_data = pd.concat([train, test], axis=0, ignore_index=True)
all_data.shape[0]

2915

In [8]:
# handle missing values
missing_features = [col for col in all_data.columns if all_data[col].isnull().any()]
len(missing_features)

34

In [9]:
# cat features --> na means none
none_features = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
    'BsmtFinType2', 'MasVnrType'
]
for col in none_features:
    all_data[col] = all_data[col].fillna('None')

In [10]:
# num features --> na means 0
zero_features = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea'
]
for col in zero_features:
    all_data[col] = all_data[col].fillna(0)

In [11]:
# LotFrontage — fill with median of neighborhood 
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage']\
    .transform(lambda x: x.fillna(x.median()))

In [12]:
# fill with mode
all_data['MSZoning'] = all_data.groupby('MSSubClass')['MSZoning']\
    .transform(lambda x: x.fillna(x.mode()[0]))

In [13]:
# fill with mode
cat_cols_mode = ['Electrical', 'KitchenQual', 'Exterior1st',
                 'Exterior2nd', 'SaleType', 'Functional']
for col in cat_cols_mode:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

In [14]:
# non predictive feature
all_data.drop(['Utilities'], axis=1, inplace=True)

In [15]:
# check  for any missing values
all_data.isnull().sum().sum()

np.int64(0)

In [16]:
#Fix Skewed Numerical Features

num_features = all_data.select_dtypes(include=[np.number]).columns

# Calculate skewness
skewed = all_data[num_features].apply(lambda x: x.skew()).sort_values(ascending=False)

# Features with skewness above threshold need log transform
skewness_threshold = 0.75
high_skew = skewed[skewed > skewness_threshold].index

print(f"Features with skewness > {skewness_threshold}: {len(high_skew)}")
print(high_skew.tolist())

# Apply log1p transform to all highly skewed features
for col in high_skew:
    all_data[col] = np.log1p(all_data[col])

# Verify
skewed_after = all_data[high_skew].apply(lambda x: x.skew())
print(f"\nMax skewness after transform: {skewed_after.max():.4f}")

Features with skewness > 0.75: 20
['MiscVal', 'PoolArea', 'LotArea', 'LowQualFinSF', '3SsnPorch', 'KitchenAbvGr', 'BsmtFinSF2', 'EnclosedPorch', 'ScreenPorch', 'BsmtHalfBath', 'MasVnrArea', 'OpenPorchSF', 'WoodDeckSF', 'MSSubClass', '1stFlrSF', 'LotFrontage', 'GrLivArea', 'BsmtFinSF1', 'BsmtUnfSF', '2ndFlrSF']

Max skewness after transform: 16.3406


In [17]:
# ── Total Square Footage ──
# The model doesn't know 1stFlr + 2ndFlr + Bsmt = total size
all_data['TotalSF'] = (all_data['TotalBsmtSF'] +
                        all_data['1stFlrSF'] +
                        all_data['2ndFlrSF'])

# ── Total Bathrooms ──
all_data['TotalBathrooms'] = (all_data['FullBath'] +
                               0.5 * all_data['HalfBath'] +
                               all_data['BsmtFullBath'] +
                               0.5 * all_data['BsmtHalfBath'])

# ── House Age when sold ──
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']

# ── Years since remodel ──
all_data['YearsSinceRemodel'] = all_data['YrSold'] - all_data['YearRemodAdd']

# ── Was the house remodeled? ──
all_data['Remodeled'] = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)

# ── Total Porch Area ──
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] +
                             all_data['EnclosedPorch'] +
                             all_data['3SsnPorch'] +
                             all_data['ScreenPorch'])

# ── Has Pool ──
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)

# ── Has Garage ──
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)

# ── Has Basement ──
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)

# ── Has Fireplace ──
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)

# ── Overall Score — quality × condition ──
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']

# ── Garage Score ──
all_data['GarageScore'] = all_data['GarageCars'] * all_data['GarageArea']

print(f"New features added. Dataset shape: {all_data.shape}")

# Verify new features make sense
print("\nNew features sample:")
new_features = ['TotalSF', 'TotalBathrooms', 'HouseAge',
                'YearsSinceRemodel', 'TotalPorchSF', 'OverallScore']
print(all_data[new_features].describe())

New features added. Dataset shape: (2915, 90)

New features sample:
           TotalSF  TotalBathrooms     HouseAge  YearsSinceRemodel  \
count  2915.000000     2915.000000  2915.000000        2915.000000   
mean   1057.853281        2.205871    36.521784          23.553002   
std     427.282950        0.806065    30.335130          20.894517   
min       5.814131        1.000000    -1.000000          -2.000000   
25%     802.469086        1.500000     7.000000           4.000000   
50%     996.898715        2.000000    35.000000          15.000000   
75%    1309.172425        2.500000    55.000000          43.000000   
max    5103.536211        7.000000   136.000000          60.000000   

       TotalPorchSF  OverallScore  
count   2915.000000    2915.00000  
mean       3.605946      33.71012  
std        2.897307       9.16474  
min        0.000000       1.00000  
25%        0.000000      30.00000  
50%        3.931826      35.00000  
75%        4.962845      40.00000  
max       16.

In [18]:
#Encode Categorical Features
# Ordinal encoding for quality features —-> order matters
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

ordinal_features = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC'
]

for col in ordinal_features:
    all_data[col] = all_data[col].map(quality_map)

In [19]:
# Label encoding for other ordinal features
from sklearn.preprocessing import LabelEncoder

label_features = ['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                  'GarageFinish', 'Fence', 'LotShape', 'LandContour',
                  'LandSlope', 'Alley']

for col in label_features:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col].astype(str))

In [20]:
# One-hot encoding for nominal features
all_data = pd.get_dummies(all_data)



In [21]:
print(f"Final dataset shape: {all_data.shape}")

Final dataset shape: (2915, 238)


In [22]:
ntrain = train.shape[0]
ntest = test.shape[0]
ntrain,ntest

(1456, 1459)

In [23]:
# Split back into train and test
X_train = all_data[:ntrain]
X_test = all_data[ntrain:]


In [24]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y shape: {y.shape}")


X_train shape: (1456, 238)
X_test shape: (1459, 238)
y shape: (1456,)


In [25]:
# Final check — no missing values
assert X_train.isnull().sum().sum() == 0, "Missing values in train!"
assert X_test.isnull().sum().sum() == 0, "Missing values in test!"
print("No missing values---> Data is clean.")


No missing values---> Data is clean.


In [26]:
# Save processed data
X_train.to_csv('/home/hossamhamdy/AI_Portfolio/Project_01_House_Price_Prediction/data/X_train_processed.csv', index=False)
X_test.to_csv('/home/hossamhamdy/AI_Portfolio/Project_01_House_Price_Prediction/data/X_test_processed.csv', index=False)
y.to_csv('/home/hossamhamdy/AI_Portfolio/Project_01_House_Price_Prediction/data/y_train.csv', index=False)